## Session 10, in class: Reiter's method

Computational Macro (HWS 2026), University of Mannheim. Thursday, 12.11.2026, 10:15 to 11:45.

This is the code behind the session 10 slides. The economy has households with CARA utility, persistent income
risk, and a constant probability of death (perpetual youth), and a firm that rents their savings as capital. It has
an aggregate version of five equations that is exact (a PRANK), so we know the answer before we start. Part 1 solves
that aggregate version with the code of week 9. Part 2 solves the steady state of the heterogeneous agent economy on
the grid, with the tools of weeks 5 to 8, and compares it with the analytical policy and distribution. Part 3 is
Reiter's method: the histogram and the marginal values on the grid become variables of one large system
$\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t) = \mathbf{0}$, which we linearize and solve with Klein, as last week. Part 4
compares the impulse responses with those of the aggregate version, part 5 with the same economy without income
risk.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **8 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: most of them are functions from weeks 5 to 9, and the
  comments say where they changed.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Slide 14" and the like refer to the session 10 slides, by the number at the bottom of each slide.
  Keep them open next to the notebook.
- Part 3 takes a few seconds per cell: the system has 801 variables. The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The aggregate economy (PRANK): steady state, linearization, Klein | 1, 2 | 4 to 7 | 10 min |
| 2 | The steady state on the grid: EGM on marginal values, Young with newborns, against the analytical solution | 3, 4 | 8 to 11 | 25 min |
| 3 | Reiter's method: the system with 801 variables, its linearization, Klein | 5, 6 | 12 to 17 | 30 min |
| 4 | Impulse responses: Reiter against the PRANK | 7 | 18 | 10 min |
| 5 | With and without income risk, and experiments | 8 | 19 | 15 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $a_i$, $e_m$ | asset and income grid points | `gri.a`, `gri.e` |
| $x = \frac{R}{\varphi} a + w (1 + e)$ | cash on hand: the annuity return on assets plus income | `x` |
| $v = \frac{R}{\varphi} \, u'(c)$ | the marginal value of assets on the grid, in logs | `lnv`, an $N_a \times N_e$ table |
| $a'(a_i, e_m)$, $c(a_i, e_m)$ | the savings and consumption policies on the grid | `a_prime`, `c` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda}^0$ | the histogram over $(a_i, e_m)$, and the one of the newborns | `lam`, `lam_born` |
| $\mathbf{Q}$ | the policy chain of week 6: survivors move from cell to cell | `Q` |
| $\mathbf{x}_t = (\boldsymbol{\lambda}_t, K_t, Z_t, \mathbf{v}_t)$ | the variables of Reiter's system, states first | `x`, a vector of length 801 |
| $\mathbf{A}$, $\mathbf{B}$, $\mathbf{G}$, $\mathbf{H}$ | as in week 9 | `A`, `B`, `G`, `H` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `np.concatenate([a, [b], c])` | one long vector from several pieces | the vector $\mathbf{x}_t$ |
| `np.append(v, s)` | the vector `v` with the number `s` added at the end | the last bin of the histogram |
| `x[:N - 1]`, `x[N + 1:]` | the first $N - 1$ entries, everything from entry $N + 1$ on | taking $\mathbf{x}_t$ apart |
| `ordqz(A, B, ..., output="real")` | the generalized Schur decomposition in real arithmetic, about four times faster | Klein with 801 variables |
| `norm.cdf(z)` from `scipy.stats` | the CDF $\Phi(z)$ of the standard normal distribution | the analytical distribution |

Everything else is from the earlier notebooks: `@dataclass`, `qe.rouwenhorst`, `np.interp`, `sp.csr_matrix`, and the
functions `jacobian`, `linearize`, `solve_klein`, `impulse_response` of week 9.

### 0. Packages

Nothing new to install: everything is in the course environment since week 5.
The first cell installs the packages into the environment of the running kernel if they are missing.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "scipy==1.15.*" "matplotlib==3.10.*" "quantecon>=0.7"

In [ ]:
import time
import warnings
from dataclasses import dataclass

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import quantecon as qe
from scipy.linalg import ordqz, inv, solve
from scipy.stats import norm

np.set_printoptions(precision=4, suppress=True)           # four decimals, no scientific notation, in every print
warnings.filterwarnings("ignore", message="The API of rouwenhorst")   # the notice on every rouwenhorst call, as in week 5

### 1. The aggregate economy: a PRANK with five equations (gaps 1 and 2)

*Slides 4 to 7.*

**Households.** A continuum of households of mass one. Each period a household survives with probability
$\varphi$, and every household that dies is replaced by a newborn without assets (perpetual youth, slide 4). Utility is
CARA, $u(c) = -e^{-\gamma c} / \gamma$, there is no borrowing limit, and income is $w_t (1 + e_t)$ with
$e_{t+1} = \rho_e e_t + \varepsilon_{t+1}$. Savings are annuities: the assets of the dead are paid out to the
survivors, so that one unit saved pays $R_t / \varphi$ with $R_t = 1 + r_t$. The household's problem is the one of
problem set 2 with $\beta \varphi$ in place of $\beta$ and $R / \varphi$ in place of $R$, and its policy is affine:
$c = \mu x + \nu w e + f$ with cash on hand $x = \frac{R}{\varphi} a + w (1 + e)$ (slide 6).

**Firm and TFP.** $Y_t = Z_t K_t^\alpha$, labor is one, so $R_t = \alpha Z_t K_t^{\alpha - 1} + 1 - \delta$ and
$w_t = (1 - \alpha) Z_t K_t^\alpha$. Log TFP is an AR(1) with persistence $\rho_Z$ around $\ln \bar Z$.

**The aggregate version (slide 7).** Since $\mu_t$ is the same for every household, aggregate consumption
depends on aggregates only. Five equations in $\mathbf{x}_t = (K_t, Z_t, C_t, \mu_t, \nu_t)$, states first:

$$
\begin{aligned}
K_{t+1} &= Z_t K_t^\alpha + (1 - \delta) K_t - C_t && \text{resource constraint} \\
\ln Z_{t+1} - \ln \bar Z &= \rho_Z (\ln Z_t - \ln \bar Z) && \text{TFP} \\
C_{t+1} &= C_t + \tfrac{1}{\gamma} \ln (\beta R_{t+1}) - \tfrac{1 - \varphi}{\varphi} \mu_{t+1} R_{t+1} K_{t+1}
          + \tfrac{\gamma}{2} (\mu_{t+1} + \nu_{t+1})^2 w_{t+1}^2 \sigma_\varepsilon^2 && \text{aggregate Euler equation} \\
\mu_t &= \frac{\mu_{t+1} R_{t+1}}{\varphi + \mu_{t+1} R_{t+1}} && \text{the MPC} \\
\nu_t &= \rho_e \frac{\varphi}{R_{t+1}} (\mu_{t+1} + \nu_{t+1}) \frac{w_{t+1}}{w_t} && \text{the loading on } e
\end{aligned}
$$

with $\sigma_\varepsilon^2 = \sigma_e^2 (1 - \rho_e^2)$ the variance of the innovation. **Calibration:** $\beta$ and
$\bar Z$ are chosen so that the steady state has $r = 4$ percent and $Y = 1$. In the steady state the MPC is the
one of problem set 2 with $R / \varphi$ in place of $R$, and the aggregate Euler equation with $C_{t+1} = C_t$
gives $\beta$.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    alpha: float = 0.36       # capital share
    delta: float = 0.1        # depreciation rate
    gamma: float = 4.0        # absolute risk aversion, CARA utility u(c) = -exp(-gamma c) / gamma
    phi: float = 44 / 45      # survival probability: 45 periods of life on average
    rho_e: float = 0.9        # persistence of the income shock e
    sig_e: float = 0.1        # unconditional standard deviation of e
    rho_Z: float = 0.9        # persistence of log TFP
    beta: float = 0.97        # discount factor, calibrated below
    Zbar: float = 1.0         # steady state TFP, calibrated below

In [ ]:
def prank_steady_state(par, r=0.04, Y=1.0):
    # The steady state of the aggregate economy with the rate r and output Y (slide 7): the firm gives K, Zbar, w,
    # the household's coefficients mu and nu, and the aggregate Euler equation with C' = C gives beta
    R = 1 + r
    K = par.alpha * Y / (r + par.delta)                                            # the firm: r = alpha Y / K - delta
    Zbar = Y / K ** par.alpha
    w = (1 - par.alpha) * Y
    C = Y - par.delta * K                                                          # the resource constraint, K' = K
    # Gap 1: the MPC out of cash on hand: problem set 2's mu = 1 - 1/R, with the annuity return R / phi in
    #        place of R
    mu = ___
    nu = par.rho_e * par.phi * mu / (R - par.rho_e * par.phi)                      # the extra response to a persistent e
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)                               # variance of the innovation to e
    ln_betaR = par.gamma * ((1 - par.phi) / par.phi * mu * R * K - par.gamma / 2 * (mu + nu) ** 2 * w ** 2 * sig_eps2)
    return dict(K=K, C=C, Z=Zbar, R=R, w=w, mu=mu, nu=nu, beta=np.exp(ln_betaR) / R, Y=Y)

In [ ]:
ss = prank_steady_state(EconomicParameters())
par = EconomicParameters(beta=ss["beta"], Zbar=ss["Z"])                         # the calibrated economy
for key, val in ss.items():
    print(f"{key:>4}: {val:.6f}")

**Check (slide 5):** $K = 2.5714$, $C = 0.7429$, $\bar Z = 0.7118$, $w = 0.64$, $\mu = 0.0598$, $\nu = 0.3291$, and
$\beta = 0.9747$. Note $\beta R = 1.014 > 1$: an infinitely lived household would let its consumption grow without
bound. Here the newborns, who own nothing, hold aggregate consumption down.

**The system and its linearization.** `F_prank` writes the five equations as residuals, with the variables in logs as
in week 9. The functions below it are the ones of week 9, unchanged except for one word in `solve_klein`: the
decomposition is done in real arithmetic (`output="real"`), which gives the same $\mathbf{G}$ and $\mathbf{H}$ and
is about four times faster. That does not matter for five variables, it does for 801 (slide 16).

In [ ]:
def prices(K, Z, par):
    # The firm's first-order conditions: the gross return R = 1 + r and the wage w
    R = par.alpha * Z * K ** (par.alpha - 1) + 1 - par.delta
    w = (1 - par.alpha) * Z * K ** par.alpha
    return R, w

In [ ]:
def F_prank(x_next, x, par):
    # The five equations of slide 7 as residuals. x = (ln K, ln Z, ln C, ln mu, ln nu) today, x_next tomorrow
    K, Z, C, mu, nu = np.exp(x)
    K1, Z1, C1, mu1, nu1 = np.exp(x_next)
    R, w = prices(K, Z, par)
    R1, w1 = prices(K1, Z1, par)
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)
    return np.array([
        np.log(K1) - np.log(Z * K ** par.alpha + (1 - par.delta) * K - C),        # resource constraint
        np.log(Z1 / par.Zbar) - par.rho_Z * np.log(Z / par.Zbar),                  # TFP, without the shock
        # Gap 2: the term of the newborns in the aggregate Euler equation (slide 7): they own none of K'
        C1 - C - np.log(par.beta * R1) / par.gamma + ___
        - par.gamma / 2 * (mu1 + nu1) ** 2 * w1 ** 2 * sig_eps2,                   # aggregate Euler equation
        np.log(mu) - np.log(mu1 * R1 / (par.phi + mu1 * R1)),                      # the MPC
        np.log(nu) - np.log(par.rho_e * par.phi / R1 * (mu1 + nu1) * w1 / w),      # the loading on e
    ])

In [ ]:
# Check: all five residuals are zero at the steady state
xss_p = np.log([ss["K"], ss["Z"], ss["C"], ss["mu"], ss["nu"]])
print("F at the steady state:", F_prank(xss_p, xss_p, par))

In [ ]:
def jacobian(f, x0, h=1e-6):
    # The Jacobian of f at x0 by central differences: column j is df / dx_j (week 9)
    n, m = len(x0), len(f(x0))
    J = np.zeros((m, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = h
        J[:, j] = (f(x0 + e) - f(x0 - e)) / (2 * h)
    return J


def linearize(F, xss, par):
    # The matrices A and B of the linear system A E x' = B x at the steady state xss (week 9)
    A = jacobian(lambda x_next: F(x_next, xss, par), xss)
    B = -jacobian(lambda x: F(xss, x, par), xss)
    return A, B

In [ ]:
def solve_klein(A, B, n_states):
    # Klein (2000), as in week 9, with one change: output="real", the same G and H, about four times faster
    S, T, alpha, beta, Q, Z = ordqz(A, B, sort=lambda a, b: np.abs(b) < np.abs(a), output="real")
    n_stable = np.sum(np.abs(beta) < np.abs(alpha))
    if n_stable != n_states:
        print(f"Klein: {n_stable} stable eigenvalues for {n_states} states")
    Z11, Z21 = Z[:n_states, :n_states], Z[n_states:, :n_states]
    S11, T11 = S[:n_states, :n_states], T[:n_states, :n_states]
    G = Z21 @ inv(Z11)                                                             # the controls on the states
    H = Z11 @ solve(S11, T11) @ inv(Z11)                                           # the law of motion of the states
    with np.errstate(divide="ignore", invalid="ignore"):
        eigs = beta / alpha                                                        # our lambda_i = t_ii / s_ii
    return G, H, eigs


def impulse_response(G, H, n_states, T, shock):
    # The paths of states and controls for T periods after the shock vector hits the states in period 0 (week 9)
    s = np.zeros((T, n_states))
    s[0] = shock
    for t in range(T - 1):
        s[t + 1] = H @ s[t]
    return s, s @ G.T

In [ ]:
A_p, B_p = linearize(F_prank, xss_p, par)
G_p, H_p, eigs_p = solve_klein(A_p, B_p, 2)
print("eigenvalues:", np.sort(np.abs(eigs_p)))
print("G (rows C, mu, nu on K, Z) =\n", G_p)
print("H =\n", H_p)
T = 60
s_p, c_p = impulse_response(G_p, H_p, 2, T, np.array([0.0, 0.01]))               # a one percent TFP shock
print(f"C on impact {100 * c_p[0, 0]:.3f} percent, K peaks at {100 * s_p[:, 0].max():.3f} percent in period {s_p[:, 0].argmax()}")

**Check (slide 7):** all five residuals are zero at the steady state, two stable eigenvalues for two states,
$0.900 = \rho_Z$ and $0.927$, the capital root, and three unstable ones, $1.052$, $1.129$, $1.187$. Consumption rises by
$0.552$ percent on impact, capital peaks at $1.027$ percent in period $11$. These are the numbers that 801 variables
have to reproduce in part 4.

### 2. The steady state on the grid (gaps 3 and 4)

*Slides 8 to 11.*

Now the same economy with the households on the grid, as in weeks 5 to 8: Rouwenhorst for $e$ with $N_e = 5$ points,
and $N_a = 80$ quadratically spaced asset points on $[-3, 50]$ (no borrowing limit, so the grid has to reach where
households go). Two things change against week 8 (slide 8).

**1. The EGM iterates on marginal values.** Week 5's `egm_step` took the consumption policy of tomorrow. Here it takes
the marginal value of assets on the grid, $v(a, e) = \frac{R}{\varphi} u'(c(a, e))$ (the envelope condition), in
logs. One step: the Euler equation $u'(c) = \beta \varphi \, \mathbb{E}[v' \mid e]$ gives consumption at every
$(a'_l, e_m)$, the budget constraint the cash on hand at which $a'_l$ is chosen, and the interpolation back to the
grid the savings policy. With CARA, $\ln v = \ln (R / \varphi) - \gamma c$: the log marginal value is consumption,
up to a constant. The reason to iterate on $v$: $v_{t+1}$ contains tomorrow's return, so that one step only needs
today's prices. That is what Reiter's system in part 3 uses.

**2. Newborns.** After the survivors have moved along the policy chain $\mathbf{Q}$ of week 6, the dead, a share
$1 - \varphi$ of the population, are replaced by newborns with $a = 0$ and $e$ drawn from its stationary distribution:
$\boldsymbol{\lambda}' = \varphi \mathbf{Q}^\top \boldsymbol{\lambda} + (1 - \varphi) \boldsymbol{\lambda}^0$.

In [ ]:
@dataclass(frozen=True)
class NumericalParameters:
    na: int = 80              # asset grid points N_a
    ne: int = 5               # income grid points N_e
    amin: float = -3.0        # lowest asset grid point: no borrowing limit, the grid has to reach where households go
    amax: float = 50.0        # highest asset grid point
    crit_pol: float = 1e-11   # the EGM stops when ln v moves by less: Reiter needs a tight steady state (slide 15)
    crit_dist: float = 1e-13  # the distribution iteration stops when lam moves by less
    maxiter: int = 100_000    # iteration cap, the safety net


@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    e: np.ndarray             # income grid, N_e
    Pi: np.ndarray            # transition matrix of e, N_e x N_e
    lam_e: np.ndarray         # stationary distribution of e, N_e


def make_grids(par, mpar):
    # Rouwenhorst for e (week 5): qe.rouwenhorst wants the standard deviation of the innovation, sig_e sqrt(1 - rho_e^2),
    # and the quadratically spaced asset grid of week 6
    mc = qe.rouwenhorst(mpar.ne, par.rho_e, par.sig_e * np.sqrt(1 - par.rho_e ** 2))
    s = np.linspace(0.0, 1.0, mpar.na)
    a = mpar.amin + (mpar.amax - mpar.amin) * s ** 2
    return Grids(a=a, e=mc.state_values, Pi=mc.P, lam_e=mc.stationary_distributions[0])

In [ ]:
mpar = NumericalParameters()
gri = make_grids(par, mpar)
print("income states e:", gri.e, " stationary distribution:", gri.lam_e, " mean", round(gri.lam_e @ gri.e, 12))
print(f"asset grid: {mpar.na} points on [{gri.a[0]}, {gri.a[-1]}], spacing at 0: {np.diff(gri.a)[np.searchsorted(gri.a, 0) - 1]:.3f}")

In [ ]:
def interp_extrap(x, xp, fp):
    # np.interp, but linear extrapolation beyond the ends of the grid instead of a flat continuation (week 9)
    y = np.interp(x, xp, fp)
    lo, hi = x < xp[0], x > xp[-1]
    y[lo] = fp[0] + (x[lo] - xp[0]) * (fp[1] - fp[0]) / (xp[1] - xp[0])
    y[hi] = fp[-1] + (x[hi] - xp[-1]) * (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return y

In [ ]:
def egm_step(lnv_next, R, w, par, gri):
    # One step of the EGM on marginal values (slide 8): from ln v' on the grid (a'_l, e_n) to today's savings policy a_prime,
    # consumption c, and ln v, all N_a x N_e, at today's prices R and w
    Ev = par.phi * np.exp(lnv_next) @ gri.Pi.T                                    # E[v' | e_m] at every a'_l, the dead need no assets
    # Gap 3: (a) the Euler equation u'(c) = beta E[v'] with u'(c) = exp(-gamma c), solved for c
    ctilde = ___
    xtilde = ctilde + gri.a[:, None]                                               # (b) the cash on hand at which a'_l is chosen
    x = R / par.phi * gri.a[:, None] + w * (1 + gri.e[None, :])                    # cash on hand on the grid
    a_prime = np.empty_like(x)
    for m in range(len(gri.e)):                                                    # (c) back to the grid, one income state at a time
        a_prime[:, m] = interp_extrap(x[:, m], xtilde[:, m], gri.a)
    c = x - a_prime
    # Gap 4: (d) the envelope condition v = (R / phi) u'(c), in logs
    lnv = ___
    return a_prime, c, lnv

In [ ]:
def solve_household(R, w, par, mpar, gri, lnv0=None):
    # Iterate egm_step at constant prices until ln v stops moving (week 5). The guess: keep assets constant, a' = a
    if lnv0 is None:
        x = R / par.phi * gri.a[:, None] + w * (1 + gri.e[None, :])
        lnv0 = np.log(R / par.phi) - par.gamma * (x - gri.a[:, None])
    lnv = lnv0
    for it in range(1, mpar.maxiter + 1):
        a_prime, c, lnv_new = egm_step(lnv, R, w, par, gri)
        dist = np.max(np.abs(lnv_new - lnv))
        lnv = lnv_new
        if dist < mpar.crit_pol:
            break
    return a_prime, c, lnv, it

**Young's method with newborns.** `lottery` and `transition_matrix` are the functions of week 8, with the income
grid now called `e`. The newborns are a histogram of mass one: assets $0$, put on the two grid points around $0$ by
the same lottery, and income from the stationary distribution of the chain. `forward` moves the histogram by one
period, `stationary_dist` iterates it until it stops moving (week 6).

In [ ]:
def lottery(a_prime, gri):
    # For every policy value a': the index l of the grid point just below it, and the weight w on the point above (week 6)
    na = len(gri.a)
    l = np.clip(np.searchsorted(gri.a, a_prime, side="right") - 1, 0, na - 2)
    w = np.clip((a_prime - gri.a[l]) / (gri.a[l + 1] - gri.a[l]), 0.0, 1.0)
    return l, w


def transition_matrix(a_prime, gri):
    # The policy chain Q of week 6: Q[(i, m), (l, n)] = Pi[m, n] (1 - w_im) and Q[(i, m), (l + 1, n)] = Pi[m, n] w_im
    na, ne = len(gri.a), len(gri.e)
    idx = np.arange(na * ne).reshape(ne, na).T                                     # idx[i, m] = i + m N_a
    l, w = lottery(a_prime, gri)
    rows, cols, vals = [], [], []
    for m in range(ne):
        for n in range(ne):
            rows.append(idx[:, m]); cols.append(idx[l[:, m], n]); vals.append(gri.Pi[m, n] * (1 - w[:, m]))
            rows.append(idx[:, m]); cols.append(idx[l[:, m] + 1, n]); vals.append(gri.Pi[m, n] * w[:, m])
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(na * ne, na * ne))

In [ ]:
def newborns(gri):
    # The newborns as a histogram of mass one: a = 0 by the lottery, e from its stationary distribution
    l, w = lottery(np.zeros((1, 1)), gri)
    lam_born = np.zeros((len(gri.a), len(gri.e)))
    lam_born[l[0, 0]] = (1 - w[0, 0]) * gri.lam_e
    lam_born[l[0, 0] + 1] = w[0, 0] * gri.lam_e
    return lam_born


def forward(lam, Q, lam_born, par):
    # One period of the histogram (slide 8): survivors move along the policy chain, the dead are replaced by newborns
    na, ne = lam.shape
    survivors = Q.T @ lam.T.ravel()                                                # week 6: lam' = Q^T lam, as a vector
    # the dead, a share 1 - phi of all households (lam sums to one), come back as newborns
    lam_new = par.phi * survivors + (1 - par.phi) * lam.sum() * lam_born.T.ravel()
    return lam_new.reshape(ne, na).T


def stationary_dist(Q, lam_born, par, mpar, lam0=None):
    # Iterate forward until the histogram stops moving (week 6), from the newborns or from lam0
    lam = lam_born.copy() if lam0 is None else lam0
    for it in range(1, mpar.maxiter + 1):
        lam_new = forward(lam, Q, lam_born, par)
        dist = np.max(np.abs(lam_new - lam))
        lam = lam_new
        if dist < mpar.crit_dist:
            break
    return lam, it

In [ ]:
lam_born = newborns(gri)
print("newborns: mass", lam_born.sum(), " mean assets", (lam_born * gri.a[:, None]).sum())

**The equilibrium.** At a rate $r$ the firm demands $K(r) = (\alpha \bar Z / (r + \delta))^{1 / (1 - \alpha)}$, which
also fixes the wage. The households supply what they save, $S(r) = \sum_{i, m} \lambda_{im} \, a'(a_i, e_m)$: today's
savings are tomorrow's capital. Supply increases in $r$, so we bisect on $S(r) - K(r)$ as in weeks 6 to 8, with warm
starts. The calibration has $r = 4$ percent in the aggregate economy, so that is where the bisection should land.

In [ ]:
@dataclass
class Aggregates:
    S: float                  # asset supply: mean savings of the households
    K: float                  # capital demand of the firm
    R: float                  # gross return 1 + r
    w: float                  # wage
    a_prime: np.ndarray       # savings policy a'(a, e), N_a x N_e
    c: np.ndarray             # consumption policy
    lnv: np.ndarray           # log marginal values
    lam: np.ndarray           # stationary histogram


def aggregate(r, par, mpar, gri, lam_born, warm=None):
    # The household block at the rate r (weeks 7 and 8): prices, policy, histogram, supply S(r) and demand K(r)
    K = (par.alpha * par.Zbar / (r + par.delta)) ** (1 / (1 - par.alpha))          # the firm's capital demand at r
    R, w = prices(K, par.Zbar, par)
    a_prime, c, lnv, _ = solve_household(R, w, par, mpar, gri, None if warm is None else warm.lnv)
    lam, _ = stationary_dist(transition_matrix(a_prime, gri), lam_born, par, mpar, None if warm is None else warm.lam)
    return Aggregates(S=(lam * a_prime).sum(), K=K, R=R, w=w, a_prime=a_prime, c=c, lnv=lnv, lam=lam)


def equilibrium(par, mpar, gri, lam_born, lo, hi, tol=1e-12):
    # Bisection on the excess supply S(r) - K(r), which increases in r, with warm starts (week 8)
    warm, k = None, 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        warm = aggregate(mid, par, mpar, gri, lam_born, warm)
        lo, hi = (mid, hi) if warm.S < warm.K else (lo, mid)
        k += 1
    r = (lo + hi) / 2
    return r, aggregate(r, par, mpar, gri, lam_born, warm), k

In [ ]:
t0 = time.perf_counter()
r_star, agg, k = equilibrium(par, mpar, gri, lam_born, 0.03, 0.05)
print(f"r* = {100 * r_star:.5f} percent after {k} bisection steps, {time.perf_counter() - t0:.1f} s")
print(f"supply S = {agg.S:.6f}, demand K = {agg.K:.6f}   (aggregate economy: K = {ss['K']:.6f})")
print(f"C = sum lam c = {(agg.lam * agg.c).sum():.6f}   (aggregate economy: C = {ss['C']:.6f})")
print(f"mass {agg.lam.sum():.12f}, at the top of the grid {agg.lam[-1].sum():.1e}, at the bottom {agg.lam[0].sum():.1e}")

**Check (slide 10):** $r^* = 4.00012$ percent, $K = 2.57140$ against $2.57143$, $C = 0.74286$ against $0.74286$. The
grid economy has five income states instead of a normal distribution, which changes the precautionary saving a
little: that is the $10^{-6}$ in $r^*$. Almost no mass at the ends of the grid ($10^{-7}$ at the top). If $r^*$ is
far from $4$ percent, check gaps 3 and 4.

**Against the analytical solution.** The policy should be $c = \mu x + \nu w e + f$ with the coefficients of part 1
and $f = C - \mu X$, $X = R K + w$ (slide 6). For the distribution, the cash on hand of a household of age $k$ is
normal with a mean $m_k$ and a variance $V_k$ that follow simple recursions, so the stationary CDF is a mixture of
normals over ages, $F(x) = (1 - \varphi) \sum_{k \geq 0} \varphi^k \, \Phi\bigl( (x - m_k) / \sqrt{V_k} \bigr)$
(slide 11, the recursions are in the backup slides). `analytical_cdf` evaluates it.

In [ ]:
def analytical_cdf(x, par, ss, tol=1e-13):
    # The stationary CDF of cash on hand (slide 11): a mixture over ages k of normals N(m_k, V_k), weights (1 - phi) phi^k
    Rphi, w, mu, nu = ss["R"] / par.phi, ss["w"], ss["mu"], ss["nu"]
    f = ss["C"] - mu * (ss["R"] * ss["K"] + w)                                     # the intercept of the policy
    kappa = par.rho_e - Rphi * nu                                                  # how much of e stays in x
    sig_eps2 = par.sig_e ** 2 * (1 - par.rho_e ** 2)
    m, V, Cov = w, w ** 2 * par.sig_e ** 2, w * par.sig_e ** 2                     # age 0: x = w (1 + e), mean, variance, cov(x, e)
    d = w - Rphi * f                                                               # the mean grows by d per year of age
    F, weight = np.zeros_like(x), 1 - par.phi
    while weight > tol:
        F += weight * norm.cdf((x - m) / np.sqrt(V))
        V, Cov = (V + w ** 2 * kappa ** 2 * par.sig_e ** 2 + w ** 2 * sig_eps2 + 2 * w * kappa * Cov,
                  par.rho_e * Cov + w * (par.rho_e * kappa * par.sig_e ** 2 + sig_eps2))
        m += d
        weight *= par.phi
    return F

In [ ]:
x_grid = agg.R / par.phi * gri.a[:, None] + agg.w * (1 + gri.e[None, :])       # cash on hand of every cell
X = ss["R"] * ss["K"] + ss["w"]
c_ana = ss["mu"] * x_grid + ss["nu"] * ss["w"] * gri.e[None, :] + ss["C"] - ss["mu"] * X
print(f"policy: largest |c - c_analytical| {np.max(np.abs(agg.c - c_ana)):.1e}, weighted by the histogram {np.sum(agg.lam * np.abs(agg.c - c_ana)):.1e}")
print(f"slope of c in x: {np.polyfit(x_grid[:, 2], agg.c[:, 2], 1)[0]:.6f}, mu = {ss['mu']:.6f}")
order = np.argsort(x_grid.ravel())                                                 # the cells sorted by cash on hand
xs, cdf_num = x_grid.ravel()[order], np.cumsum(agg.lam.ravel()[order])
cdf_ana = analytical_cdf(xs, par, ss)
print(f"E[x] = {np.sum(agg.lam * x_grid):.5f} against X = R K + w = {X:.5f}")
print(f"largest gap between the two CDFs: {np.max(np.abs(cdf_num - cdf_ana)):.3f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.3))
axs[0].plot(gri.a, agg.c[:, 2], lw=2, label="grid, $e = 0$")
axs[0].plot(gri.a, c_ana[:, 2], "--", color="k", lw=1, label="analytical")
axs[0].set_xlabel("assets $a$"); axs[0].set_ylabel("consumption"); axs[0].legend(); axs[0].set_xlim(-3, 20)
axs[1].plot(xs, cdf_num, lw=2, label=f"Young, $N_a = {mpar.na}$")
axs[1].plot(xs, cdf_ana, "--", color="k", lw=1, label="analytical")
axs[1].set_xlabel("cash on hand $x$"); axs[1].set_ylabel("CDF"); axs[1].legend(); axs[1].set_xlim(-1, 20)
plt.tight_layout(); plt.show()

**Check (slides 10 and 11):** the policy is within $10^{-4}$ of the analytical one ($2 \cdot 10^{-5}$ weighted by the
histogram), the slope of $c$ in $x$ is $\mu = 0.05983$ to five digits, and $\mathbb{E}[x] = 3.31425$ against $X = 3.31429$: the lottery preserves means.
The CDFs differ by up to $0.07$ on this grid. That is the lottery: it puts the mass of a household on the two grid
points around its choice, which preserves the mean but adds variance. The mean, and with it every aggregate, is right
on any grid. On finer grids the gap shrinks (slide 11), experiment 2 at the end of the notebook lets you check it.

### 3. Reiter's method: the histogram and the marginal values as variables (gaps 5 and 6)

*Slides 12 to 17.*

Reiter's idea: on the grid, the distribution is a vector, and the steps that computed the steady state are
equations. The variables of period $t$ are, states first (slide 13),

| Block | Entries | Type | Deviation |
| --- | --- | --- | --- |
| histogram $\boldsymbol{\lambda}_t$, all bins but the last | $N_a N_e - 1 = 399$ | state | level |
| capital $K_t$ | $1$ | state | log |
| TFP $Z_t$ | $1$ | state | log |
| marginal values $\ln \mathbf{v}_t$ on the grid | $N_a N_e = 400$ | control | log |

and $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t)$ has four blocks (slide 12): Young's step forward, the asset market
$K_{t+1} = \sum \lambda_t \, a'_t$, TFP, and the EGM step backward,
$\ln \mathbf{v}_t = \ln (R_t / \varphi) - \gamma \mathbf{c}_t$, where $\mathbf{c}_t$ and $\mathbf{a}'_t$ come from
one `egm_step` on $\ln \mathbf{v}_{t+1}$ at today's prices (slide 8 draws the histogram as a vector and Young's step as a matrix product). In
the code the savings policy is called `a_prime`: in weeks 6 to 8 it was `G`, which here is Klein's $\mathbf{G}$. The
histogram is a state: it is given at the start of
the period. The marginal values are controls: they look forward, like consumption in week 9.

**Why one bin is missing (slide 15).** The histogram sums to one, so one bin is redundant: it is one minus the
others. Kept as a variable, it adds an eigenvalue of exactly one, which is neither stable nor explosive, and
Klein's count fails. **Levels for the histogram:** bins can be empty, and $\ln 0$ does not
exist. Levels are also the baseline of the BASEforHANK toolbox of Bayer, Born, and Luetticke. Everything else is in
logs, as in week 9.

In [ ]:
def unpack(x, gri):
    # The vector x of slide 13 as its four blocks: the histogram (with its last bin restored), K, Z, and ln v
    na, ne = len(gri.a), len(gri.e)
    N = na * ne
    # Gap 5: the last bin: the histogram sums to one
    lam_vec = np.append(x[:N - 1], ___)
    lam = lam_vec.reshape(ne, na).T                                                # back to the N_a x N_e table of week 6
    lnv = x[N + 1:].reshape(ne, na).T
    return lam, np.exp(x[N - 1]), np.exp(x[N]), lnv


def pack(lam, K, Z, lnv):
    # The inverse of unpack: the four blocks as one vector, the histogram without its last bin
    N = lam.size
    return np.concatenate([lam.T.ravel()[:N - 1], [np.log(K), np.log(Z)], lnv.T.ravel()])

In [ ]:
def F_reiter(x_next, x, par, gri, lam_born):
    # Reiter's system (slides 12 and 14): zero along an equilibrium path without shocks
    lam, K, Z, lnv = unpack(x, gri)                                                # today
    lam1, K1, Z1, lnv1 = unpack(x_next, gri)                                       # tomorrow
    R, w = prices(K, Z, par)                                                       # today's prices
    a_prime, c, lnv_egm = egm_step(lnv1, R, w, par, gri)                                 # one EGM step from tomorrow's v
    lam_fwd = forward(lam, transition_matrix(a_prime, gri), lam_born, par)              # one step of Young's method
    N = lam.size
    return np.concatenate([
        (lam1 - lam_fwd).T.ravel()[:N - 1],                                        # Young: all bins but the last
        # Gap 6: the asset market: tomorrow's capital is the mean of today's savings, in logs
        [___],
        [np.log(Z1 / par.Zbar) - par.rho_Z * np.log(Z / par.Zbar)],                # TFP, without the shock
        (lnv - lnv_egm).T.ravel(),                                                 # EGM: today's marginal values
    ])

In [ ]:
N = mpar.na * mpar.ne
xss = pack(agg.lam, agg.K, par.Zbar, agg.lnv)
# the number of states: the histogram without its last bin, capital, and TFP
n_states = N + 1
F_r = lambda x_next, x, par: F_reiter(x_next, x, par, gri, lam_born)               # F with the grids filled in
print(f"{len(xss)} variables, {n_states} states, {len(xss) - n_states} controls")
print(f"largest residual at the steady state: {np.max(np.abs(F_r(xss, xss, par))):.1e}")

**Check (slides 13 and 15):** $801$ variables, $401$ states, $400$ controls, and a residual below $10^{-10}$ at
the steady state. A residual of $10^{-5}$ or more means the steady state is not the one of $\mathbf{F}$: the
linearization would be around a point that is not a steady state. (At $r = 4$ percent instead of $r^*$ the residual
is $7 \cdot 10^{-5}$, try it with `aggregate(0.04, ...)`.)

**Linearize and solve.** The same two lines as in week 9. `linearize` evaluates $\mathbf{F}$ $4 \times 801$ times,
`solve_klein` decomposes two $801 \times 801$ matrices: a few seconds each (slide 16).

In [ ]:
t0 = time.perf_counter()
A, B = linearize(F_r, xss, par)
t_jac = time.perf_counter() - t0
t0 = time.perf_counter()
G, H, eigs = solve_klein(A, B, n_states)
t_qz = time.perf_counter() - t0
print(f"Jacobians: {t_jac:.1f} s, Klein: {t_qz:.1f} s")
mods = np.sort(np.abs(eigs))
print(f"stable eigenvalues: {np.sum(mods < 1)} for {n_states} states")
print("the five largest stable:", mods[mods < 1][-5:])
print("the five smallest explosive:", mods[mods > 1][:5])

In [ ]:
plt.figure(figsize=(7, 3))
plt.semilogy(np.arange(1, len(mods) + 1), np.where(np.isfinite(mods), mods, np.nan), ".", ms=3)
plt.axhline(1, color="k", lw=0.8); plt.axvline(n_states + 0.5, color="gray", ls=":")
plt.xlabel("eigenvalues, sorted by modulus"); plt.ylabel("modulus"); plt.ylim(1e-6, 1e3)
plt.title(f"{np.sum(mods < 1)} stable, {np.sum(mods > 1)} explosive (infinite ones not shown)", fontsize=10)
plt.tight_layout(); plt.show()

**Check (slides 16 and 17):** $401$ stable eigenvalues for $401$ states: the count fits, one bounded solution. The
largest stable ones sit just below $\varphi = 0.978$: the histogram forgets its past at the speed at which households
die. $\rho_Z = 0.9$ is among them. The smallest explosive one is $1.051$. On a laptop the Jacobians take a few
seconds, Klein between two and ten.

### 4. Impulse responses: Reiter against the PRANK (gap 7)

*Slide 18.*

A one percent TFP shock: the state vector starts at zero everywhere except for log TFP, and $\mathbf{H}$ and
$\mathbf{G}$ do the rest, exactly as in week 9. Capital and TFP are states, output follows from them,
$\hat Y_t = \hat Z_t + \alpha \hat K_t$. Consumption we compute twice: from the resource constraint,
$C_t = Y_t + (1 - \delta) K_t - K_{t+1}$, and from the households, $C_t = \sum \lambda_t c_t$, with the change of the
policy read off the marginal values, $\ln v = \ln (R / \varphi) - \gamma c$ gives
$dc = (d \ln R - d \ln v) / \gamma$. If the goods market clears, the two agree.

In [ ]:
s0 = np.zeros(n_states)
s0[N] = 0.01                                                                       # log TFP is the last state
s_h, c_h = impulse_response(G, H, n_states, T, s0)                                 # states and controls, T x 401 and T x 400
K_h, Z_h = s_h[:, N - 1], s_h[:, N]                                                # log deviations of K and Z
Y_h = Z_h + par.alpha * K_h
K, Y = agg.K, par.Zbar * agg.K ** par.alpha
C = (agg.lam * agg.c).sum()
C_rc = (Y * Y_h[:-1] + (1 - par.delta) * K * K_h[:-1] - K * K_h[1:]) / C           # from the resource constraint, T - 1 periods

In [ ]:
def consumption_from_households(s_h, c_h, agg, par, gri):
    # C_t = sum lam_t c_t, linearized: d lam times c plus lam times dc, with dc from the marginal values
    na, ne = len(gri.a), len(gri.e)
    N = na * ne
    dlnR = (agg.R - 1 + par.delta) / agg.R * (s_h[:, N] + (par.alpha - 1) * s_h[:, N - 1])   # d ln R from d ln Z and d ln K
    dC = np.zeros(len(s_h))
    for t in range(len(s_h)):
        dlam = np.append(s_h[t, :N - 1], -s_h[t, :N - 1].sum()).reshape(ne, na).T  # the change of the histogram, sums to zero
        dlnv = c_h[t].reshape(ne, na).T                                            # the change of ln v on the grid
        # Gap 7: the change of consumption on the grid, from ln v = ln(R / phi) - gamma c
        dc = ___
        dC[t] = (dlam * agg.c).sum() + (agg.lam * dc).sum()
    return dC / (agg.lam * agg.c).sum()

In [ ]:
C_hh = consumption_from_households(s_h, c_h, agg, par, gri)
print(f"largest gap, capital, Reiter against the PRANK: {np.max(np.abs(K_h - s_p[:, 0])):.1e}")
print(f"largest gap, consumption from the households against the resource constraint: {np.max(np.abs(C_hh[:-1] - C_rc)):.1e}")
print(f"C on impact {100 * C_hh[0]:.3f} percent (PRANK {100 * c_p[0, 0]:.3f}), K peaks at {100 * K_h.max():.3f} percent in period {K_h.argmax()}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, (name, ha, ra) in zip(axs, (("capital", K_h, s_p[:, 0]), ("consumption", C_hh, c_p[:, 0]), ("output", Y_h, s_p[:, 1] + par.alpha * s_p[:, 0]))):
    ax.plot(100 * ha, lw=2, label="Reiter, 801 variables")
    ax.plot(100 * ra, "o", ms=3, mfc="white", color="k", label="PRANK, 5 equations")
    ax.set_title(name, fontsize=10); ax.set_xlabel("periods after the shock"); ax.axhline(0, color="k", lw=0.5)
axs[0].set_ylabel("percent deviation"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Check (slide 18):** capital within $1.5 \cdot 10^{-5}$ of the PRANK at every horizon (in log points, at a peak of
$0.0103$), consumption on impact $0.552$ percent in both, capital peaks at $1.026$ percent in period $11$. The two
consumption series agree to $10^{-8}$: the goods market clears, as Walras's law says it must. 801 variables
and five equations give the same aggregate dynamics, because the households' MPC is the same for all of them.

### 5. With and without income risk (gap 8)

*Slide 19.*

Does the heterogeneity from income risk matter for the aggregates? The clean comparison keeps everything else: the
same newborns ($\varphi = 44/45$), the same utility, the same steady state, but no income risk, $\sigma_e = 0$.
Households still differ, by age and therefore by wealth, but no longer by luck. In the five equations of slide 7 only
the precautionary term changes: it is zero. The steady state keeps $r = 4$ percent and $Y = 1$, so `prank_steady_state`
gives the same $K$, $C$, and $\bar Z$ and a new $\beta$: without risk there is no precautionary saving, and a higher
$\beta$ has to hold the capital stock at the same level. The code is the one of part 1, only the parameters differ.

In [ ]:
# Gap 8: the same economy without income risk
ss_nr = prank_steady_state(EconomicParameters(___))
par_nr = EconomicParameters(beta=ss_nr["beta"], Zbar=ss_nr["Z"], sig_e=0.0)
print(f"beta without income risk {par_nr.beta:.5f}, with income risk {par.beta:.5f}; K {ss_nr['K']:.4f} and {ss['K']:.4f}")
xss_nr = np.log([ss_nr["K"], ss_nr["Z"], ss_nr["C"], ss_nr["mu"], ss_nr["nu"]])
A_nr, B_nr = linearize(F_prank, xss_nr, par_nr)
G_nr, H_nr, eigs_nr = solve_klein(A_nr, B_nr, 2)
s_nr, c_nr = impulse_response(G_nr, H_nr, 2, T, np.array([0.0, 0.01]))
print("eigenvalues:", np.sort(np.abs(eigs_nr)))
print(f"K peaks at {100 * s_nr[:, 0].max():.3f} percent (with income risk {100 * K_h.max():.3f}), C on impact {100 * c_nr[0, 0]:.3f} percent ({100 * C_hh[0]:.3f})")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, (name, ha, nr) in zip(axs, (("capital", K_h, s_nr[:, 0]), ("consumption", C_hh, c_nr[:, 0]))):
    ax.plot(100 * ha, lw=2, label="with income risk (Reiter)")
    ax.plot(100 * nr, "--", lw=2, label="without income risk")
    ax.set_title(name, fontsize=10); ax.set_xlabel("periods after the shock"); ax.axhline(0, color="k", lw=0.5)
axs[0].set_ylabel("percent deviation"); axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

**Check (slide 19):** $\beta = 0.97563$ without income risk against $0.97471$ with it, the same $K = 2.5714$.
Capital peaks at $1.025$ percent without income risk against $1.026$ with it, consumption rises by $0.553$ percent on
impact against $0.552$. The two lines lie on top of each other. With CARA utility every household has the same MPC
$\mu$, so who is rich and who is poor does not matter for aggregate consumption. Income risk enters the aggregate
only through the precautionary term, which is a few $10^{-4}$ of consumption and moves little after the shock.

**Experiments.**

1. **More income risk.** Set $\sigma_e = 0.3$ (three times the baseline) in the aggregate economy of part 1 and
   compare with the economy without risk. Does the gap grow? Then redo parts 2 to 4 on the grid: what do you have
   to change in `NumericalParameters` so that no mass reaches the ends of the asset grid?
2. **The grid: precision against cost.** Solve parts 2 and 3 with $N_a = 40$, $120$, and $200$
   (`NumericalParameters(na=...)`). In part 2, record the largest gap between the two CDFs and the standard deviation
   of cash on hand: how fast does the distribution approach the analytical one (slide 11)? In part 3, time the
   Jacobians and Klein: how does the time scale with $n$? Do the impulse responses of capital change? (They should
   not by much: the aggregates only need the mean of the distribution, which the lottery gets right on any grid.)
   Part 2 alone also runs with $N_a = 1000$; part 3 would then have $10{,}001$ variables.
3. **Real against complex.** Put `output="complex"` (Julia: `schur(complex(A), complex(B))`) back into
   `solve_klein`, as in week 9, and compare the time and $\mathbf{G}$.